<a href="https://colab.research.google.com/github/barnalibhowmick0-dot/PGII_BB/blob/ASSIGNMENT_3/ASSIGNMEMT_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import ee

In [2]:
import geemap

In [3]:
ee.Authenticate()
ee.Initialize(project="barnali-ee")

In [5]:
import geemap as map
map

<module 'geemap' from '/usr/local/lib/python3.12/dist-packages/geemap/__init__.py'>

In [6]:
map = geemap.Map()
map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [7]:
district = ee.FeatureCollection("projects/barnali-ee/assets/DISTRICT_BOUNDARY")

In [8]:
map.addLayer(district, {}, "District_bnd")
map

Map(bottom=812.0, center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=Search…

In [9]:
MOD44B = ee.ImageCollection("MODIS/006/MOD44B").select("Percent_Tree_Cover")

In [19]:
tree_cover_2002 = MOD44B.filterDate('2002-01-01', '2002-12-31').mean().clip(district)

tree_cover_2020 = MOD44B.filterDate('2020-01-01', '2020-12-31').mean().clip(district)

vis_tree02 = {'min': 0,'max': 77,'palette': ['#f7f4f0', '#e8dcc8', '#d4c19a', '#c8e6a0', '#a8d96e', '#7cc242', '#5aaa1c', '#3d8b14', '#2d6b0f', '#1a4d0a']}
vis_tree20 = {'min': 0,'max': 77,'palette': ['#f7f4f0', '#e8dcc8', '#d4c19a', '#c8e6a0', '#a8d96e', '#7cc242', '#5aaa1c', '#3d8b14', '#2d6b0f', '#1a4d0a']}
map.addLayer(tree_cover_2002, vis_tree02, "tree_cover_2002")
map.addLayer(tree_cover_2020, vis_tree20, "tree_cover_2020")
map

Map(bottom=1964.0, center=[31.952162238024975, 89.384765625], controls=(WidgetControl(options=['position', 'tr…

In [22]:
def compute_zonal_stats(image, features, scale=500):
    """
    Compute zonal statistics for each district
    """
    def calc_stats(feature):
        clipped = image.clip(feature.geometry())
        stats = clipped.reduceRegion(
            reducer=ee.Reducer.mean().combine(
                ee.Reducer.min(), '', True).combine(
                ee.Reducer.max(), '', True).combine(
                ee.Reducer.stdDev(), '', True),
            geometry=feature.geometry(),
            scale=scale,
            maxPixels=1e13
        )
        return feature.set(stats)

    return features.map(calc_stats)


districts_tree_2002 = compute_zonal_stats(tree_cover_2002, district, scale=250)
districts_tree_2020 = compute_zonal_stats(tree_cover_2020, district, scale=250)


In [58]:
import geemap

geemap.ee_to_geojson(districts_tree_2002, filename='MOD44B_TreeCover_2002.geojson')
print("Exported: MOD44B_TreeCover_2002.geojson")

geemap.ee_to_geojson(districts_tree_2020, filename='MOD44B_TreeCover_2020.geojson')
print("Exported: MOD44B_TreeCover_2020.geojson")

Exported: MOD44B_TreeCover_2002.geojson
Exported: MOD44B_TreeCover_2020.geojson


In [25]:
MCD15A3H = ee.ImageCollection("MODIS/061/MCD15A3H").select("Lai")

In [30]:
lai_2005 = MCD15A3H.filterDate('2005-01-01', '2005-12-31').median().multiply(0.1).clip(district)
lai_2023 = MCD15A3H.filterDate('2023-01-01', '2023-12-31').median().multiply(0.1).clip(district)
vis_lai02 = {'min': 0, 'max': 5.3, 'palette': ['#f7fcf5', '#e7f6e2', '#ceecc7', '#aedea7', '#88cd86', '#5db96b', '#37a055', '#1b843f', '#00682a', '#00441b']}
vis_lai23 = {'min': 0, 'max': 5.3, 'palette': ['#f7fcf5', '#e7f6e2', '#ceecc7', '#aedea7', '#88cd86', '#5db96b', '#37a055', '#1b843f', '#00682a', '#00441b']}
map.addLayer(lai_2005, vis_lai02, "lai_2002")
map.addLayer(lai_2023, vis_lai23, "lai_2020")
map

Map(bottom=2087.0, center=[22.350075806124867, 100.01953125000001], controls=(WidgetControl(options=['position…

In [32]:
def compute_zonal_stats(image, features, scale=500):
    """
    Compute zonal statistics for each district
    """
    def calc_stats(feature):
        clipped = image.clip(feature.geometry())
        stats = clipped.reduceRegion(
            reducer=ee.Reducer.mean().combine(
                ee.Reducer.min(), '', True).combine(
                ee.Reducer.max(), '', True).combine(
                ee.Reducer.stdDev(), '', True),
            geometry=feature.geometry(),
            scale=scale,
            maxPixels=1e13
        )
        return feature.set(stats)

    return features.map(calc_stats)

districts_lai_2005 = compute_zonal_stats(lai_2005, district, scale=500)
districts_lai_2023 = compute_zonal_stats(lai_2023, district, scale=500)

In [57]:
import geemap

geemap.ee_to_geojson(districts_lai_2005, filename='MCD15A3H_LAI_2005.geojson')
print("Exported: MCD15A3H_LAI_2005.geojson")

geemap.ee_to_geojson(districts_lai_2023, filename='MCD15A3H_LAI_2023.geojson')
print("Exported: MCD15A3H_LAI_2023.geojson")

Exported: MCD15A3H_LAI_2005.geojson
Exported: MCD15A3H_LAI_2023.geojson


In [33]:
MOD11A1 = ee.ImageCollection("MODIS/061/MOD11A1").select("LST_Day_1km")

In [34]:
lst_2006 = MOD11A1.filterDate('2006-01-01', '2006-12-31').mean().multiply(0.02).subtract(273.15).clip(district)

lst_2022 = MOD11A1.filterDate('2022-01-01', '2022-12-31').mean().multiply(0.02).subtract(273.15).clip(district)

vis_lst06 = {'min': 15, 'max': 45, 'palette': ['#2166ac', '#4393c3', '#92c5de', '#d1e5f0', '#fef8b8', '#fee08b', '#fdae61', '#f46d43', '#d73027', '#a50026']}
vis_lst22 = {'min': 15, 'max': 45, 'palette': ['#2166ac', '#4393c3', '#92c5de', '#d1e5f0', '#fef8b8', '#fee08b', '#fdae61', '#f46d43', '#d73027', '#a50026']}
map.addLayer(lst_2006, vis_lst06, "lst_2006")
map.addLayer(lst_2022, vis_lst22, "lst_2022")
map

Map(bottom=2117.0, center=[19.89072302399691, 78.70259809618507], controls=(WidgetControl(options=['position',…

In [35]:
def compute_zonal_stats(image, features, scale=500):
    """
    Compute zonal statistics for each district
    """
    def calc_stats(feature):
        clipped = image.clip(feature.geometry())
        stats = clipped.reduceRegion(
            reducer=ee.Reducer.mean().combine(
                ee.Reducer.min(), '', True).combine(
                ee.Reducer.max(), '', True).combine(
                ee.Reducer.stdDev(), '', True),
            geometry=feature.geometry(),
            scale=scale,
            maxPixels=1e13
        )
        return feature.set(stats)

    return features.map(calc_stats)

# Compute zonal statistics for LST
districts_lst_2006 = compute_zonal_stats(lst_2006, district, scale=1000)
districts_lst_2022 = compute_zonal_stats(lst_2022, district, scale=1000)


In [56]:
import geemap

geemap.ee_to_geojson(districts_lst_2006, filename='MOD11A1_LST_2006.geojson')
print("Exported: MOD11A1_LST_2006.geojson")

geemap.ee_to_geojson(districts_lst_2022, filename='MOD11A1_LST_2022.geojson')
print("Exported: MOD11A1_LST_2022.geojson")

Exported: MOD11A1_LST_2006.geojson
Exported: MOD11A1_LST_2022.geojson
